# 최종 단계: Stacking_LR 재학습 및 모델 저장

> 이전 실험 기록: 이 파일은 TabICL을 포함한 기존 백엔드 모델 `deal-stacking-lr-v1`의 재학습·저장 과정이다. 최신 `deal-paper-rf-ensemble-v1` 후보의 저장 단계가 아니며 자동으로 이어 실행하지 않는다. 현재 실행 순서는 [노트북 안내](README.md)를 따른다.

확정된 `Stacking_LR` 구성을 전체 448건의 마스킹 데이터로 재학습하고 배포용 아티팩트를 저장한다.

1. 3단계에서 선택한 다섯 기본 모델의 파라미터를 고정한다.
2. 전체 448건에 만든 10개 마스킹 세트에서 반복 Group OOF 확률을 생성한다.
3. 다섯 OOF 확률로 LogisticRegression 메타모델을 학습한다.
4. 다섯 기본 모델을 전체 4,480행으로 다시 학습한다.
5. 모델·스키마·임계값·메타데이터를 저장하고 새 CPU 프로세스에서 재로드한다.

평가 수치는 4단계의 Train/Test 분리 결과를 유지한다. 이 노트북의 전체 데이터 재학습은 배포 모델을 만드는 단계다.

## 0. 실행 환경

```bash
uv sync --project backend/notebooks --locked
uv run --project backend/notebooks --locked jupyter lab backend/notebooks/deal_model_finalization.ipynb
```

원본 CSV 위치가 기본값과 다르면 `SALESLUV_B2B_DATA_PATH` 환경변수로 지정한다. TabICL은 MPS, CUDA, CPU 순으로 사용 가능한 장치를 선택한다.

In [1]:
import hashlib
import json
import platform
import subprocess
import sys
import warnings
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from catboost import CatBoostClassifier
from IPython import get_ipython
from IPython.utils.capture import capture_output
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from tabicl import TabICLClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)

current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    notebooks_dir = current_dir
    repository_root = current_dir.parents[1]
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    notebooks_dir = current_dir / "notebooks"
    repository_root = current_dir.parent
elif (current_dir / "backend" / "notebooks").is_dir():
    repository_root = current_dir
    notebooks_dir = current_dir / "backend" / "notebooks"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

preprocessing_notebook = notebooks_dir / "deal_data_preprocessing.ipynb"
artifact_dir = repository_root / "backend" / "pipeline" / "artifacts"
phase3_selection_path = artifact_dir / "deal_phase3_selection.joblib"
phase4_prediction_path = artifact_dir / "deal_phase4_predictions.joblib"

assert preprocessing_notebook.exists()
assert phase3_selection_path.exists()
assert phase4_prediction_path.exists()

ipython = get_ipython()
assert ipython is not None, "이 파일은 Jupyter에서 실행해야 합니다."
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

MODEL_FEATURE_NAMES = ipython.user_ns["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = ipython.user_ns["CATEGORY_VALUES"]
X_all_masked_sets = ipython.user_ns["X_all_masked_sets"]
y = ipython.user_ns["y"]
input_group_ids = ipython.user_ns["input_group_ids"]
SOURCE_SHA256 = ipython.user_ns["SOURCE_SHA256"]
UNKNOWN_COLUMNS_PER_ROW = ipython.user_ns["UNKNOWN_COLUMNS_PER_ROW"]

if torch.backends.mps.is_available():
    tabicl_device = "mps"
elif torch.cuda.is_available():
    tabicl_device = "cuda"
else:
    tabicl_device = "cpu"

print(f"저장소 루트: {repository_root}")
print(f"TabICL 장치: {tabicl_device}")

데이터 전처리 검증을 통과했습니다.
저장소 루트: .
TabICL 장치: mps


### 해석

- 전처리·선택·평가 아티팩트가 모두 존재할 때만 최종 재학습을 시작한다.
- TabICL 장치를 자동 확인하고 사용된 장치를 메타데이터에 기록한다.

In [2]:
MODEL_VERSION = "deal-stacking-lr-v1"
CLASSIFICATION_THRESHOLD = 0.5
OUTER_SEEDS = (1, 11, 21)
OUTER_FOLDS = 5
BASE_MODEL_ORDER = (
    "LogisticRegression",
    "MultinomialNB",
    "ExtraTrees",
    "CatBoost",
    "TabICL",
)

selection_artifact = joblib.load(phase3_selection_path)
phase4_artifact = joblib.load(phase4_prediction_path)

assert selection_artifact["model_feature_names"] == list(MODEL_FEATURE_NAMES)
assert phase4_artifact["selected_candidate"] == "Stacking_LR"
assert phase4_artifact["classification_threshold"] == CLASSIFICATION_THRESHOLD
best_params = selection_artifact["best_params"]


def make_one_hot_model(classifier):
    encoder = OneHotEncoder(
        categories=[list(CATEGORY_VALUES[column]) for column in MODEL_FEATURE_NAMES],
        drop="first",
        handle_unknown="error",
        sparse_output=False,
        dtype=np.float32,
    )
    return Pipeline([("onehot", encoder), ("classifier", classifier)])


base_models = {
    "LogisticRegression": make_one_hot_model(
        LogisticRegression(max_iter=3000, solver="lbfgs", random_state=OUTER_SEEDS[0])
    ),
    "MultinomialNB": make_one_hot_model(MultinomialNB()),
    "ExtraTrees": make_one_hot_model(ExtraTreesClassifier(random_state=OUTER_SEEDS[0], n_jobs=1)),
    "CatBoost": CatBoostClassifier(
        cat_features=tuple(MODEL_FEATURE_NAMES),
        loss_function="Logloss",
        verbose=False,
        allow_writing_files=False,
        random_seed=OUTER_SEEDS[0],
        thread_count=1,
    ),
    "TabICL": TabICLClassifier(
        batch_size=8,
        kv_cache=False,
        allow_auto_download=True,
        device=tabicl_device,
        use_fa3="auto",
        offload_mode="auto",
        random_state=OUTER_SEEDS[0],
        n_jobs=1,
        verbose=False,
    ),
}
for model_name, model in base_models.items():
    model.set_params(**best_params[model_name])

assert tuple(base_models) == BASE_MODEL_ORDER
print("3단계 최적 파라미터로 다섯 기본 모델을 구성했습니다.")

3단계 최적 파라미터로 다섯 기본 모델을 구성했습니다.


### 해석

- 모델 종류와 하이퍼파라미터는 3단계 선택 결과로 고정한다.
- One-Hot Encoder의 범주와 순서는 1단계 입력 스키마를 그대로 사용한다.
- 메타모델의 입력 순서는 `LogisticRegression → MultinomialNB → ExtraTrees → CatBoost → TabICL`이다.

In [3]:
# 전체 448건의 열 개 마스킹 세트를 하나의 최종 학습 데이터로 합친다.
X_final_raw = pd.concat(
    X_all_masked_sets,
    names=["mask_set", "original_row_id"],
)
y_final = pd.concat(
    {set_name: y for set_name in X_all_masked_sets},
    names=["mask_set", "original_row_id"],
).astype("int8")
final_original_row_ids = X_final_raw.index.get_level_values("original_row_id")
final_group_ids = input_group_ids.loc[final_original_row_ids].to_numpy()

assert X_final_raw.shape == (4480, 13)
assert y_final.shape == (4480,)
assert X_final_raw.index.equals(y_final.index)
assert len(np.unique(final_group_ids)) == 198
assert all(str(dtype) == "category" for dtype in X_final_raw.dtypes)
assert X_final_raw.eq("Unknown").sum(axis=1).eq(4).all()

print(f"최종 학습 입력: {X_final_raw.shape}")
print(f"원본 행: {len(y)}, 동일 입력 그룹: {len(np.unique(final_group_ids))}")
print(f"Lost/Won: {(y_final == 0).sum()} / {(y_final == 1).sum()}")

최종 학습 입력: (4480, 13)
원본 행: 448, 동일 입력 그룹: 198
Lost/Won: 2210 / 2270


### 해석

- 평가가 끝난 448개 원본 행을 모두 사용해 배포 모델을 학습한다.
- 한 원본 행마다 서로 다른 마스킹 10개가 있어 최종 학습 행은 4,480개다.
- 같은 입력 그룹의 변형은 OOF Fold를 넘지 않도록 198개 그룹 ID를 유지한다.

In [4]:
def positive_class_probability(estimator, X):
    won_index = list(estimator.classes_).index(1)
    return estimator.predict_proba(X)[:, won_index]


# 최종 Stacking 메타모델이 볼 확률은 Group OOF 예측으로 만든다.
repeat_base_oof = []
for repeat_number, seed in enumerate(OUTER_SEEDS, start=1):
    group_cv = StratifiedGroupKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=seed,
    )
    splits = list(group_cv.split(X_final_raw, y_final, groups=final_group_ids))
    for train_index, valid_index in splits:
        assert set(final_group_ids[train_index]).isdisjoint(set(final_group_ids[valid_index]))

    repeat_probability = np.full(
        (len(y_final), len(BASE_MODEL_ORDER)),
        np.nan,
        dtype=float,
    )
    for model_index, (model_name, model) in enumerate(base_models.items()):
        model_jobs = 1 if model_name == "TabICL" else -1
        probabilities = cross_val_predict(
            clone(model),
            X_final_raw,
            y_final,
            groups=final_group_ids,
            cv=splits,
            n_jobs=model_jobs,
            method="predict_proba",
        )
        repeat_probability[:, model_index] = probabilities[:, 1]
        print(f"반복 {repeat_number}/{len(OUTER_SEEDS)} {model_name} OOF 완료")

    assert np.isfinite(repeat_probability).all()
    repeat_base_oof.append(repeat_probability)

repeat_base_oof = np.stack(repeat_base_oof)
mean_base_oof = repeat_base_oof.mean(axis=0)
assert mean_base_oof.shape == (4480, 5)
print("최종 메타모델용 반복 Group OOF 확률을 생성했습니다.")

반복 1/3 LogisticRegression OOF 완료


반복 1/3 MultinomialNB OOF 완료


반복 1/3 ExtraTrees OOF 완료


반복 1/3 CatBoost OOF 완료


반복 1/3 TabICL OOF 완료


...

반복 2/3 TabICL OOF 완료


반복 3/3 LogisticRegression OOF 완료
반복 3/3 MultinomialNB OOF 완료


반복 3/3 ExtraTrees OOF 완료


반복 3/3 CatBoost OOF 완료


반복 3/3 TabICL OOF 완료
최종 메타모델용 반복 Group OOF 확률을 생성했습니다.


### 해석

- 세 개 시드의 5-Fold Group OOF 확률을 평균해 메타모델 학습 입력을 만든다.
- 각 행의 OOF 확률은 그 행과 같은 입력 그룹을 학습하지 않은 기본 모델이 계산한다.
- 이 단계는 최종 모델의 결합 학습용이며, 보고 성능은 4단계의 분리 평가 결과를 사용한다.

In [5]:
final_stacking_model = LogisticRegression(
    C=0.1,
    max_iter=3000,
    solver="lbfgs",
    random_state=OUTER_SEEDS[0],
).fit(mean_base_oof, y_final)

fitted_base_models = {}
for model_name, model in base_models.items():
    fitted_model = clone(model)
    if model_name == "TabICL":
        fitted_model.set_params(kv_cache="repr")
    fitted_base_models[model_name] = fitted_model.fit(X_final_raw, y_final)
    print(f"{model_name} 전체 학습 완료")


def synthetic_self_check_record(*known_column_indexes):
    """원본 행을 복사하지 않고 허용 범주만 조합한 계약 검증 입력을 만든다."""
    record = {column: "Unknown" for column in MODEL_FEATURE_NAMES}
    for column_index in known_column_indexes:
        column = MODEL_FEATURE_NAMES[column_index]
        known_values = [value for value in CATEGORY_VALUES[column] if value != "Unknown"]
        record[column] = known_values[column_index % len(known_values)]
    return record


self_check_X = pd.DataFrame(
    [
        synthetic_self_check_record(),
        *(synthetic_self_check_record(index) for index in range(len(MODEL_FEATURE_NAMES))),
        synthetic_self_check_record(0, 1),
        synthetic_self_check_record(2, 3),
    ],
    columns=MODEL_FEATURE_NAMES,
)
assert len(self_check_X) == 16
assert (self_check_X == "Unknown").sum(axis=1).min() >= 11
self_check_base_probability = np.column_stack(
    [
        positive_class_probability(fitted_base_models[model_name], self_check_X)
        for model_name in BASE_MODEL_ORDER
    ]
)
expected_probabilities = positive_class_probability(
    final_stacking_model,
    self_check_base_probability,
)

assert np.isfinite(expected_probabilities).all()
assert ((expected_probabilities >= 0) & (expected_probabilities <= 1)).all()
print("전체 4,480행 최종 학습과 기준 확률 계산을 완료했습니다.")

LogisticRegression 전체 학습 완료
MultinomialNB 전체 학습 완료


ExtraTrees 전체 학습 완료


CatBoost 전체 학습 완료


TabICL 전체 학습 완료


전체 4,480행 최종 학습과 기준 확률 계산을 완료했습니다.


### 해석

- 메타모델은 반복 OOF 확률로 학습했다.
- 다섯 기본 모델은 배포 추론을 위해 전체 4,480행으로 다시 학습했다.
- 저장 전 기준 확률 16개를 만들어 재로드 결과와 비교한다.

In [6]:
MODEL_BUNDLE_PATH = artifact_dir / f"{MODEL_VERSION}-models.joblib"
TABICL_MODEL_PATH = artifact_dir / f"{MODEL_VERSION}-tabicl.pkl"
MODEL_METADATA_PATH = artifact_dir / f"{MODEL_VERSION}.json"
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib_bundle = {
    "schema_version": 1,
    "model_version": MODEL_VERSION,
    "base_model_order": list(BASE_MODEL_ORDER),
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "category_values": {column: list(values) for column, values in CATEGORY_VALUES.items()},
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "base_models": {
        name: fitted_base_models[name] for name in BASE_MODEL_ORDER if name != "TabICL"
    },
    "stacking_model": final_stacking_model,
}
joblib.dump(joblib_bundle, MODEL_BUNDLE_PATH, compress=3)
tabicl_artifact = fitted_base_models["TabICL"]
tabicl_artifact.device = "cpu"
tabicl_artifact.save(
    TABICL_MODEL_PATH,
    save_model_weights=True,
    save_training_data=False,
    save_kv_cache=True,
)


def file_sha256(path):
    """파일 내용을 청크 단위로 읽어 SHA-256을 계산한다."""
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def metrics(y_true, probability):
    """확률 예측에서 평가 지표와 혼동행렬 수치를 계산한다."""
    prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "brier": brier_score_loss(y_true, probability),
        "logloss": log_loss(y_true, probability, labels=[0, 1]),
        "auc": roc_auc_score(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


cv_repeat_metrics = []
for repeat_probability in phase4_artifact["cv_probabilities"]:
    mask_metrics = []
    labels = phase4_artifact["cv_mask_set_labels"]
    for mask_set in np.unique(labels):
        selected = labels == mask_set
        mask_metrics.append(
            metrics(phase4_artifact["cv_target"][selected], repeat_probability[selected])
        )
    cv_repeat_metrics.append(pd.DataFrame(mask_metrics).mean().to_dict())

test_set_metrics = [
    metrics(phase4_artifact["test_target"], probability)
    for probability in phase4_artifact["test_probabilities"].values()
]
cv_metric_frame = pd.DataFrame(cv_repeat_metrics)
test_metric_frame = pd.DataFrame(test_set_metrics)

metadata = {
    "schema_version": 1,
    "model_version": MODEL_VERSION,
    "selected_model": "Stacking_LR",
    "base_model_order": list(BASE_MODEL_ORDER),
    "stacking": {
        "meta_model": "LogisticRegression",
        "C": 0.1,
        "training_probabilities": "3 repeated 5-fold StratifiedGroupKFold OOF mean",
    },
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "category_values": {column: list(values) for column, values in CATEGORY_VALUES.items()},
    "target": {"Lost": 0, "Won": 1},
    "data": {
        "source_sha256": SOURCE_SHA256,
        "raw_rows": len(y),
        "masked_sets": len(X_all_masked_sets),
        "masked_training_rows": len(y_final),
        "input_groups": len(np.unique(final_group_ids)),
        "unknown_columns_per_row": UNKNOWN_COLUMNS_PER_ROW,
        "uses_all_raw_rows": True,
    },
    "evaluation_before_full_refit": {
        "cv_mean": cv_metric_frame.mean().to_dict(),
        "cv_brier_std": float(cv_metric_frame["brier"].std(ddof=0)),
        "test_mean": test_metric_frame.mean().to_dict(),
        "test_brier_std": float(test_metric_frame["brier"].std(ddof=1)),
        "test_mask_sets": len(test_metric_frame),
    },
    "best_params": best_params,
    "files": {
        "models": {
            "path": MODEL_BUNDLE_PATH.name,
            "format": "joblib",
            "size_bytes": MODEL_BUNDLE_PATH.stat().st_size,
            "sha256": file_sha256(MODEL_BUNDLE_PATH),
        },
        "tabicl": {
            "path": TABICL_MODEL_PATH.name,
            "format": "tabicl-native-with-weights-and-repr-cache",
            "size_bytes": TABICL_MODEL_PATH.stat().st_size,
            "sha256": file_sha256(TABICL_MODEL_PATH),
        },
    },
    "versions": {
        "python": platform.python_version(),
        "joblib": version("joblib"),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "catboost": version("catboost"),
        "tabicl": version("tabicl"),
        "torch": torch.__version__,
    },
    "training_device": {"tabicl": tabicl_device},
    "serialization_device": {"tabicl": "cpu"},
    "persistence": {"tabicl": {"training_data_included": False, "kv_cache": "repr"}},
    "saved_at_utc": datetime.now(UTC).isoformat(),
    "self_check": {
        "records": self_check_X.astype("string").to_dict(orient="records"),
        "won_probabilities": expected_probabilities.tolist(),
        "rtol": 1e-5,
        "atol": 1e-6,
    },
}
MODEL_METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(f"모델 번들: {MODEL_BUNDLE_PATH}")
print(f"TabICL: {TABICL_MODEL_PATH}")
print(f"메타데이터: {MODEL_METADATA_PATH}")

모델 번들: backend/pipeline/artifacts/deal-stacking-lr-v1-models.joblib
TabICL: backend/pipeline/artifacts/deal-stacking-lr-v1-tabicl.pkl
메타데이터: backend/pipeline/artifacts/deal-stacking-lr-v1.json


### 해석

- joblib 번들에는 One-Hot Encoder를 포함한 세 모델, CatBoost, Stacking 메타모델, 입력 스키마가 들어간다.
- TabICL은 가중치와 표현 캐시만 저장하고 원본 학습 행은 제외한다.
- JSON에는 데이터·평가·환경·파일 해시와 합성 입력 16건의 기준 확률을 기록한다.

In [7]:
# 현재 프로세스에서 한 번 재로드해 직렬화 전후 확률을 비교한다.
restored_bundle = joblib.load(MODEL_BUNDLE_PATH)
restored_tabicl = TabICLClassifier.load(TABICL_MODEL_PATH, device=tabicl_device)
restored_models = {
    **restored_bundle["base_models"],
    "TabICL": restored_tabicl,
}
restored_base_probability = np.column_stack(
    [
        positive_class_probability(restored_models[model_name], self_check_X)
        for model_name in restored_bundle["base_model_order"]
    ]
)
restored_probabilities = positive_class_probability(
    restored_bundle["stacking_model"],
    restored_base_probability,
)
np.testing.assert_allclose(
    restored_probabilities,
    expected_probabilities,
    rtol=metadata["self_check"]["rtol"],
    atol=metadata["self_check"]["atol"],
)
same_process_max_diff = float(np.max(np.abs(restored_probabilities - expected_probabilities)))
print(f"현재 프로세스 재로드 최대 확률 차이: {same_process_max_diff:.12g}")

현재 프로세스 재로드 최대 확률 차이: 0


In [8]:
# 새 CPU 프로세스에서 모델과 메타데이터를 다시 읽어 같은 기준 확률을 확인한다.
verification_code = r"""import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from tabicl import TabICLClassifier

bundle_path, tabicl_path, metadata_path = map(Path, sys.argv[1:4])
bundle = joblib.load(bundle_path)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

for key, path in (("models", bundle_path), ("tabicl", tabicl_path)):
    import hashlib
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != metadata["files"][key]["sha256"]:
        raise RuntimeError(f"{key} SHA-256 mismatch")

X = pd.DataFrame(metadata["self_check"]["records"])
X = X[metadata["model_feature_names"]]
for column, categories in metadata["category_values"].items():
    X[column] = pd.Categorical(X[column], categories=categories)

tabicl = TabICLClassifier.load(tabicl_path)
if tabicl.device != "cpu" or tabicl.device_.type != "cpu":
    raise RuntimeError("TabICL artifact is not serialized for CPU")
if (
    tabicl.kv_cache != "repr"
    or tabicl.model_kv_cache_ is None
    or tabicl.ensemble_generator_.X_ is not None
    or tabicl.ensemble_generator_.y_ is not None
):
    raise RuntimeError("TabICL artifact contains training rows or lacks repr cache")
models = {**bundle["base_models"], "TabICL": tabicl}
base_probability = []
for name in bundle["base_model_order"]:
    model = models[name]
    won_index = list(model.classes_).index(1)
    base_probability.append(model.predict_proba(X)[:, won_index])
base_probability = np.column_stack(base_probability)
meta = bundle["stacking_model"]
won_index = list(meta.classes_).index(1)
actual = meta.predict_proba(base_probability)[:, won_index]
expected = np.asarray(metadata["self_check"]["won_probabilities"], dtype=float)
np.testing.assert_allclose(
    actual,
    expected,
    rtol=metadata["self_check"]["rtol"],
    atol=metadata["self_check"]["atol"],
)
print(json.dumps({"max_abs_diff": float(np.max(np.abs(actual - expected)))}))
"""

verification = subprocess.run(
    [
        sys.executable,
        "-c",
        verification_code,
        str(MODEL_BUNDLE_PATH),
        str(TABICL_MODEL_PATH),
        str(MODEL_METADATA_PATH),
    ],
    check=True,
    capture_output=True,
    text=True,
    timeout=1800,
)
print(f"새 CPU 프로세스 검증: {verification.stdout.strip()}")

assert file_sha256(MODEL_BUNDLE_PATH) == metadata["files"]["models"]["sha256"]
assert file_sha256(TABICL_MODEL_PATH) == metadata["files"]["tabicl"]["sha256"]
assert MODEL_METADATA_PATH.exists()
print("최종 모델 저장·해시·재로드 검증을 통과했습니다.")

새 CPU 프로세스 검증: {"max_abs_diff": 1.0451534443456367e-06}
최종 모델 저장·해시·재로드 검증을 통과했습니다.


### 최종 해석

- `deal-stacking-lr-v1`의 모델 파일과 메타데이터를 CPU 이식 가능한 형태로 저장했다.
- 같은 프로세스와 새 CPU 프로세스에서 저장 전후 확률 일치 검사를 수행했다.
- 파일 SHA-256도 메타데이터와 일치한다.
- 백엔드는 JSON의 입력 스키마와 임계값을 적용하고 두 모델 파일을 함께 배포해야 한다.